# Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", ".*dubious year.*")
warnings.filterwarnings(
    "ignore", "Tried to get polar motions for times after IERS data is valid.*"
)

In [ ]:
import numpy as np
import scipy.sparse
from astropy import units as u
from astropy.coordinates import (
    SkyCoord,
    UnitSphericalRepresentation,
)
from astropy.table import QTable, join, vstack
from astropy.time import Time
from astropy.utils.masked import Masked, combine_masks
from m4opt.dynamics import EigenAxisSlew, nominal_roll
from m4opt.missions import uvex as mission
from m4opt.missions import uvex_downlink_orientation
from m4opt.utils.optimization import solve_tsp
from matplotlib import pyplot as plt
from regions import Regions
from tqdm.auto import tqdm

In [ ]:
(inscribed_fov,) = Regions.read("../fov/inscribed-circle.ds9")
fields = QTable.read("../tables/fields.ecsv")
blocks = QTable.read("../tables/skyblocks.ecsv")
fields = join(fields, blocks)
fields

## Survey Schedule

In [ ]:
downlink_duration = 30 * u.min
observe_duration = 900 * u.s

Get the times and orientations for downlinks.

In [ ]:
mission_start_time = Time("2030-01-01")
mission_duration = 2 * u.year
downlink_cadence_days = 0.25
downlink_times = (
    mission_start_time
    + np.arange(0, mission_duration.to_value(u.day), downlink_cadence_days) * u.day
)
# downlink_times = Time("2030-01-01") + np.arange(0, 102, downlink_cadence_days) * u.day
downlink_target_coords, downlink_rolls = uvex_downlink_orientation(downlink_times)
downlink_target_coords = downlink_target_coords.icrs
downlink_observer_location = mission.observer_location(downlink_times)
downlinks = QTable(
    {
        "start_time": downlink_times,
        "target_coord": SkyCoord(
            downlink_target_coords.ra,
            downlink_target_coords.dec,
            representation_type=UnitSphericalRepresentation,
        ),
        "roll": downlink_rolls,
        "observer_location": downlink_observer_location,
    }
)
downlinks["action"] = "downlink"
downlinks["duration"] = downlink_duration
downlinks.write("../tables/downlinks.ecsv", overwrite=True)

Determine the time blocks during which each sky block is fully observable.

In [ ]:
tick = 1 * u.hour
ticks = (
    mission_start_time
    + 0.5 * tick
    + np.arange(0, mission_duration.to_value(u.day), tick.to_value(u.day)) * u.day
)

observer_location = mission.observer_location(ticks)
field_tick_observable = mission.constraints(
    observer_location[np.newaxis, :],
    fields["target_coord"][:, np.newaxis],
    ticks[np.newaxis, :],
)
field_tick_observable

In [ ]:
timeblock_by_tick = (
    np.digitize(
        (ticks - mission_start_time).to_value(u.day),
        bins=(downlink_times - mission_start_time).to_value(u.day),
    )
    - 1
)
field_timeblock_observable = np.asarray(
    [
        np.logical_and.reduce(field_tick_observable[:, timeblock_by_tick == i], axis=1)
        for i in range(timeblock_by_tick.max())
    ]
).T
assert field_timeblock_observable.shape[1] == len(downlink_times) - 1
field_timeblock_observable

In [ ]:
skyblock_timeblock_observable = np.asarray(
    [
        np.logical_and.reduce(
            field_timeblock_observable[fields["block_id"] == i, :], axis=0
        )
        for i in range(fields["block_id"].max() + 1)
    ]
)
skyblock_timeblock_observable

In [ ]:
blocks_visits_goal = (
    fields["block_id", "skyblock_visits_goal"]
    .group_by("block_id")
    .groups.aggregate(np.max)
)
assert np.all(blocks_visits_goal["block_id"] == np.arange(len(blocks_visits_goal)))
blocks_visits_goal

In [ ]:
def _():
    duty_cycle = 1.75
    for row, visits in zip(
        skyblock_timeblock_observable, blocks_visits_goal["skyblock_visits_goal"]
    ):
        period = int(np.floor(len(row) / (visits + np.minimum(duty_cycle, 1) - 1)))
        num_on = int(np.ceil(duty_cycle * period))
        for i in range(visits - 1):
            mask = np.zeros(len(row), dtype=bool)
            mask[i * period : np.minimum(i * period + num_on, len(mask))] = True
            yield mask & row
        mask = np.zeros(len(row), dtype=bool)
        mask[-num_on:] = True
        yield mask & row


connections = np.asarray(list(_()))
aspect = 20
fig_width = 8
fig, ax = plt.subplots(figsize=(fig_width, aspect * fig_width))
ax.imshow(connections, interpolation="none", aspect=aspect)

In [ ]:
assignment = scipy.sparse.csgraph.maximum_bipartite_matching(
    scipy.sparse.csr_array(connections)
)
block_assignment = np.asarray(blocks_visits_goal["block_id"]).repeat(
    blocks_visits_goal["skyblock_visits_goal"]
)[assignment]
block_assignment[assignment == -1] = -1
assert np.sum(block_assignment != -1) == len(connections), "not a full matching"
block_assignment

Calculate weights for each of the blocks, equal to the total time required to reach a given depth in each field.

In [ ]:
# with observing(
#     block_observer_locations[np.newaxis, :],
#     mission.skygrid[:, np.newaxis],
#     block_times[np.newaxis, :],
# ):
#     exptime = mission.detector.get_exptime(
#         5, synphot.SourceSpectrum(synphot.ConstFlux1D, amplitude=24.5 * u.ABmag), "FUV"
#     ).to_value(u.s)
# partition_exptime = np.asarray(
#     [np.sum(exptime[partition == i, :], axis=0) for i in range(partition.max() + 1)]
# )
# partition_exptime = np.clip(partition_exptime, 0, 86400)
# partition_exptime

Find the minimum weighted matching: map each partition to the lowest-background time to observe it.

In [ ]:
# blocks = (
#     fields["block_id", "expected_visits"].group_by("block_id").groups.aggregate(np.max)
# )
# blocks.sort("block_id")
# partition_multiplicity = blocks["expected_visits"]

# graph = nx.Graph()
# graph.add_weighted_edges_from(
#     # ((i, visit), j, partition_exptime[i, j])
#     ((i, visit), j, np.random.uniform(1, 2))
#     for i, j in zip(*np.nonzero(partition_observable))
#     for visit in range(partition_multiplicity[i])
# )

# matching = nx.bipartite.minimum_weight_full_matching(
#     graph, np.arange(partition_observable.shape[1])
# )

# matching_i, matching_j = np.transpose(
#     [(matching[j][0], j) for j in range(partition_observable.shape[1]) if j in matching]
# )

In [ ]:
# graph = nx.Graph()
# graph.add_weighted_edges_from(
#     [
#         (i, j + partition_observable.shape[0], partition_exptime[i, j])
#         for i, j in zip(*np.nonzero(partition_observable))
#     ]
# )
# matching = nx.bipartite.minimum_weight_full_matching(
#     graph, np.arange(partition_observable.shape[0])
# )
# i = np.asarray(list(matching.keys()))
# j = np.asarray(list(matching.values())) - partition_observable.shape[0]
# keep = j >= 0
# i = i[keep]
# j = j[keep]
# sort = np.argsort(j)
# matching_j = j[sort]
# matching_i = i[sort]

In [ ]:
targets = QTable(
    {
        "field_id": fields["field_id"],
        "block_id": fields["block_id"],
        "target_coord": SkyCoord(
            fields["target_coord"].ra,
            fields["target_coord"].dec,
            representation_type=UnitSphericalRepresentation,
        ),
    }
)
targets["action"] = "observe"
targets["duration"] = observe_duration
targets

In [ ]:
def plan_block(downlink, targets):
    slew_targets = vstack((downlink, targets))
    slew_targets["roll"] = nominal_roll(
        slew_targets["observer_location"][0],
        slew_targets["target_coord"],
        slew_targets["start_time"][0],
    )

    slew_time = mission.slew.time(
        slew_targets["target_coord"][:, np.newaxis],
        slew_targets["target_coord"][np.newaxis, :],
        slew_targets["roll"][:, np.newaxis],
        slew_targets["roll"][np.newaxis, :],
    )
    # No slew time between identical fields
    slew_time[
        slew_targets["target_coord"][:, np.newaxis]
        == slew_targets["target_coord"][np.newaxis, :]
    ] = 0 * u.s
    if len(slew_targets) == 1:
        seq = np.zeros(1, dtype=int)
    else:
        seq, _ = solve_tsp(slew_time.to_value(u.s), verbose=False)
        assert seq[0] == 0

    # Find optimal slew path
    slews = QTable({"duration": slew_time[seq[:-1], seq[1:]]})
    slews["action"] = "slew"

    # Interleave slews with observations
    slew_targets = slew_targets[seq]
    slew_targets["i"] = np.arange(len(slew_targets))
    slews["i"] = np.arange(len(slews)) + 0.5
    plan = vstack((slew_targets, slews))
    plan.sort("i")
    del plan["i"]

    # Fill in observation times, observer locations
    plan["start_time"][1:] = plan["start_time"][0] + np.cumsum(plan["duration"][:-1])
    plan["observer_location"] = mission.observer_location(plan["start_time"])

    # Done!
    return plan


plan = vstack(
    [
        plan_block(
            downlinks[timeblock_id : timeblock_id + 1],
            targets[[] if skyblock_id == -1 else targets["block_id"] == skyblock_id],
        )[: (-1 if timeblock_id < len(block_assignment) - 1 else None)]
        for timeblock_id, skyblock_id in enumerate(tqdm(block_assignment))
    ]
)
plan

Sanity check: no actions overlap in time.

In [ ]:
end_time = plan["start_time"] + plan["duration"]
((plan["start_time"][1:] - end_time[:-1]).to(u.s)).min()

Add total time metadata.

In [ ]:
total_time_by_action = (
    plan["action", "duration"].group_by("action").groups.aggregate(np.sum)
)
plan.meta["total_time"] = {
    str(row["action"]): row["duration"] for row in total_time_by_action
}
end_time = plan["start_time"] + plan["duration"]
down_time = (plan["start_time"][1:] - end_time[:-1]).to(u.s)
assert down_time.min() >= -1e-3 * u.s
plan.meta["total_time"]["slack"] = np.maximum(down_time, 0 * u.s).sum()

Add slew angles.

In [ ]:
args = [
    plan["target_coord"][:-2],
    plan["target_coord"][2:],
    plan["roll"][:-2],
    plan["roll"][2:],
]
plan["slew_angle"] = Masked(
    np.pad(EigenAxisSlew.separation(*args), 1, constant_values=np.nan),
    np.pad(combine_masks([arg.mask for arg in args]), 1, constant_values=True),
)

In [ ]:
plan[
    "start_time",
    "duration",
    "observer_location",
    "action",
    "target_coord",
    "roll",
    "field_id",
    "block_id",
    "slew_angle",
].write("../tables/plan.ecsv", overwrite=True)

In [ ]:
# duration = plan["start_time"][-1] + plan["duration"][-1] - plan["start_time"][0]

# observations = plan[plan["action"] == "observe"]

# fig = plt.figure(dpi=300)
# ax = fig.add_subplot(projection="astro mollweide")

# frames = plan["start_time"][0] + np.linspace(0, 1, 10_000) * duration

# observable = mission.constraints(
#     mission.observer_location(frames)[:, np.newaxis],
#     observations["target_coord"],
#     frames[:, np.newaxis],
# )

# body_markers = [sun, earth, moon(-110)]
# body_positions = [
#     get_body(body, frames, mission.observer_location(frames))
#     for body in ["sun", "earth", "moon"]
# ]

# body_artists = [
#     ax.plot(
#         position[0].ra.deg,
#         position[0].dec.deg,
#         marker=marker,
#         color="red",
#         transform=ax.get_transform("world"),
#     )[0]
#     for marker, position in zip(body_markers, body_positions)
# ]

# artist = ax.scatter(
#     observations["target_coord"].ra.deg,
#     observations["target_coord"].dec.deg,
#     transform=ax.get_transform("world"),
#     linewidths=0,
# )

# with tqdm(total=len(frames)) as progress:

#     def animate(i):
#         t = frames[i]
#         keep = observations["start_time"] < t
#         artist.set_sizes(keep * 20 + 1)
#         artist.set_alpha(0.6 * (observable[i] | keep) + 0.4)
#         for body_artist, position in zip(body_artists, body_positions):
#             body_artist.set_data([position[i].ra.deg], [position[i].dec.deg])
#         progress.update()
#         return [artist, *body_artists]

#     FuncAnimation(
#         fig,
#         animate,
#         np.arange(len(frames)),
#         blit=True,
#         interval=25,
#     ).save("visualizations/plan.mp4")